<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [0]:
#| echo: false
#| output: asis
show_doc(TranscriptIndex)

---

[source](https://github.com/cobioda/allos/blob/main/allos/transcript_data.py#L54){target="_blank" style="float:right; font-size:smaller"}

### TranscriptIndex

>      TranscriptIndex (tx_by_id:Dict[str,__main__.TxRecord],
>                       gene_to_tx:Dict[str,List[str]],
>                       genename_to_tx:Dict[str,List[str]], cache_key:str)

*Fast, read-only index built from a GTF/GFF. O(1) lookups for blocks & metadata.*

In [0]:
#| echo: false
#| output: asis
show_doc(TxRecord)

---

[source](https://github.com/cobioda/allos/blob/main/allos/transcript_data.py#L32){target="_blank" style="float:right; font-size:smaller"}

### TxRecord

>      TxRecord (transcript_id:str, gene_id:Optional[str],
>                gene_name:Optional[str], transcript_name:Optional[str],
>                transcript_type:Optional[str], chrom:str, strand:int,
>                span:Tuple[int,int], exons:numpy.ndarray, cds:numpy.ndarray,
>                utr5:numpy.ndarray, utr3:numpy.ndarray)

In [0]:
#| echo: false
#| output: asis
show_doc(TranscriptData)

---

[source](https://github.com/cobioda/allos/blob/main/allos/transcript_data.py#L293){target="_blank" style="float:right; font-size:smaller"}

### TranscriptData

>      TranscriptData (gtf_file:str, reference_fasta:Optional[str]=None,
>                      cache_dir:Optional[str]=None)

*Fast, drop-in replacement for your old TranscriptData, backed by TranscriptIndex.*

In [ ]:
from pathlib import Path
import urllib.request
import pyranges as pr

# ---- set data dir (relative) ----
data_dir = Path("..") / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# ---- use the MOUSE GTF (Ensembl GRCm39 release 109) ----
mouse_gtf_url = "ftp://ftp.ensembl.org/pub/release-109/gtf/mus_musculus/Mus_musculus.GRCm39.109.gtf.gz"
mouse_gtf_local = data_dir / "Mus_musculus.GRCm39.109.gtf.gz"

# (optional) mouse primary assembly FASTA
mouse_fa_url = "ftp://ftp.ensembl.org/pub/release-109/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz"
mouse_fa_local = data_dir / "Mus_musculus.GRCm39.dna.primary_assembly.fa.gz"

# ---- download if needed ----
if not mouse_gtf_local.exists():
    print(f"Downloading mouse GTF → {mouse_gtf_local} …")
    urllib.request.urlretrieve(mouse_gtf_url, mouse_gtf_local)
# if not mouse_fa_local.exists():
#     print(f"Downloading mouse FASTA → {mouse_fa_local} …")
#     urllib.request.urlretrieve(mouse_fa_url, mouse_fa_local)

# ---- sanity check: ensure ENSMUS* transcript IDs exist ----
df_head = pr.read_gtf(str(mouse_gtf_local)).df.head(50000)
has_mouse_tids = df_head["transcript_id"].astype(str).str.startswith("ENSMUS").any()
if not has_mouse_tids:
    raise RuntimeError(f"{mouse_gtf_local} does not appear to contain mouse transcript IDs (ENSMUS…).")

# ---- point your code at the MOUSE files ----
gtf_file_local = mouse_gtf_local
fasta_file_local = mouse_fa_local  # optional

In [ ]:
# Instantiate your TranscriptData
td = TranscriptData(
    gtf_file=gtf_file_local,
    reference_fasta=fasta_file_local
)

# Example query
example_transcript_id = "ENSMUST00000070533"  # mouse
exons = td.get_exons(example_transcript_id)
print("Exons:", exons)

Exons: +--------------+-----------+-----------+--------------+------------+
|   Chromosome |     Start |       End | Strand       | Feature    |
|   (category) |   (int64) |   (int64) | (category)   | (object)   |
|--------------+-----------+-----------+--------------+------------|
|            1 |   3284704 |   3287191 | -            | exon       |
|            1 |   3491924 |   3492124 | -            | exon       |
|            1 |   3740774 |   3741721 | -            | exon       |
+--------------+-----------+-----------+--------------+------------+
Stranded PyRanges object has 3 rows and 5 columns from 1 chromosomes.
For printing, the PyRanges was sorted on Chromosome and Strand.


In [ ]:
td._idx.summary()

# %%
# See the first few transcript IDs
td._idx.list_transcripts(n=10)

['ENSMUST00000000001',
 'ENSMUST00000000003',
 'ENSMUST00000000010',
 'ENSMUST00000000028',
 'ENSMUST00000000033',
 'ENSMUST00000000049',
 'ENSMUST00000000058',
 'ENSMUST00000000080',
 'ENSMUST00000000087',
 'ENSMUST00000000090']